In [323]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import joblib

In [324]:
df = pd.read_csv("engine_failure_dataset.csv")

df['Time_Stamp'] = pd.to_datetime(df['Time_Stamp'])
df = df.sort_values('Time_Stamp')

df['vibration_mag'] = np.sqrt(
    df['Vibration_X']**2 +
    df['Vibration_Y']**2 +
    df['Vibration_Z']**2
)

df['temp_ma'] = df['Temperature (°C)'].rolling(5).mean()
df['rpm_ma'] = df['RPM'].rolling(5).mean()
df['vib_ma'] = df['vibration_mag'].rolling(5).mean()

df = df.dropna()

le = LabelEncoder()
df['Operational_Mode'] = le.fit_transform(df['Operational_Mode'])

In [325]:
features = df.drop(columns=['Fault_Condition', 'Time_Stamp'])
y = df['Fault_Condition']

In [326]:
mean = features.mean().values
cov = np.cov(features.values, rowvar=False)

n_samples = len(features)  # same size as real
synthetic_X = np.random.multivariate_normal(mean, cov, size=n_samples)
synthetic_df = pd.DataFrame(synthetic_X, columns=features.columns)

In [327]:
bounds = {
    'RPM': (500, 6000),
    'Temperature (°C)': (0, 150),
    'Fuel_Efficiency': (0, 30),
    'Vibration_X': (0, 5),
    'Vibration_Y': (0, 5),
    'Vibration_Z': (0, 5),
    'Torque': (0, 600),
    'Power_Output (kW)': (0, 300),
}

for col, (low, high) in bounds.items():
    if col in synthetic_df.columns:
        synthetic_df[col] = synthetic_df[col].clip(low, high)

In [328]:
synthetic_df['vibration_mag'] = np.sqrt(
    synthetic_df['Vibration_X']**2 +
    synthetic_df['Vibration_Y']**2 +
    synthetic_df['Vibration_Z']**2
)

synthetic_df['temp_ma'] = synthetic_df['Temperature (°C)']
synthetic_df['rpm_ma'] = synthetic_df['RPM']
synthetic_df['vib_ma'] = synthetic_df['vibration_mag']
synthetic_df['hour'] = np.random.randint(0, 24, size=len(synthetic_df))

In [329]:
synthetic_y = (
    (synthetic_df['vibration_mag'] > 2.5) |
    (synthetic_df['Temperature (°C)'] > 100) |
    (synthetic_df['RPM'] > 4500)
).astype(int)

In [330]:
X_combined = pd.concat([features, synthetic_df], ignore_index=True)
y_combined = pd.concat([y, pd.Series(synthetic_y)], ignore_index=True)

In [331]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)
iso = IsolationForest(
    n_estimators=300,
    contamination=0.02,
    random_state=42
)
iso.fit(X_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.02
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [332]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.81      0.79       196
           1       0.71      0.65      0.68       116
           2       0.33      0.35      0.34        52
           3       0.11      0.11      0.11        35

    accuracy                           0.64       399
   macro avg       0.48      0.48      0.48       399
weighted avg       0.64      0.64      0.64       399



In [333]:
feature_columns = X_combined.columns.tolist()

pipeline = {
    "iso_model": iso,
    "rf_model": rf,
    "scaler": scaler,
    "label_encoder": le,
    "feature_columns": feature_columns
}

joblib.dump(pipeline, "vehicle_pipeline.pkl")
print("✅ retrained + saved")


✅ retrained + saved
